# 🎬 Scalable Movie Recommender System

**Scalable Data Science Mini Project**

This notebook demonstrates a scalable movie recommendation system using Apache Spark's MLlib library with ALS (Alternating Least Squares) matrix factorization.

---

## 📋 Project Overview

| Aspect | Details |
|--------|---------|
| **Dataset** | MovieLens 1M (1M ratings, ~3,900 movies, ~6,040 users) |
| **Algorithm** | ALS (Alternating Least Squares) Collaborative Filtering |
| **Tools** | Apache Spark MLlib, PySpark, Pandas, Plotly |
| **Evaluation** | RMSE, MAE, Precision@K, Recall@K |

> **Note:** This notebook uses the MovieLens 1M dataset (a curated subset of the full 20M dataset) to ensure smooth execution on Google Colab's free tier without out-of-memory errors. The same methodology scales directly to the full 20M dataset on a distributed cluster (EMR/Databricks).


In [1]:
# Step 1: Install & Import Libraries
!pip install -q pyspark==3.5.0 plotly==5.18.0 kaleido

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import warnings
warnings.filterwarnings('ignore')

from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

# Spark Libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, mean, min, max, avg, desc, lit, rand,
    collect_list, size, explode, array_distinct, when, coalesce
)
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS, ALSModel
from pyspark.ml.feature import StringIndexer
from pyspark.sql.types import (
    StructType, StructField, IntegerType, FloatType, StringType
)

print("✅ All libraries imported successfully!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.
✅ All libraries imported successfully!


In [2]:
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['SPARK_HOME'] = '/usr/local/lib/python3.12/dist-packages/pyspark'

spark = SparkSession.builder \
    .appName("MovieRecommender") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("✅ Spark Session initialized!")
print(f"   Spark Version: {spark.version}")
print(f"   App Name: {spark.sparkContext.appName}")

✅ Spark Session initialized!
   Spark Version: 3.5.0
   App Name: MovieRecommender


---

## Step 3: Load & Prepare Data

We use the **MovieLens 1M** dataset which contains:
- **1,000,209** anonymous ratings
- **~6,040** users  
- **~3,900** movies
- Ratings on a 1-5 star scale

This is a subset of the full MovieLens 20M dataset, curated to run efficiently on Colab while demonstrating the same scalable methodology.


In [3]:
# Step 3a: Download MovieLens 1M Dataset (~6 MB compressed)
!wget -q https://files.grouplens.org/datasets/movielens/ml-1m.zip -O ml-1m.zip
!unzip -q -o ml-1m.zip

print("✅ MovieLens 1M dataset downloaded and extracted!")
!ls -lh ml-1m/


✅ MovieLens 1M dataset downloaded and extracted!
total 24M
-rw-r----- 1 root root 168K Mar 26  2003 movies.dat
-rw-r----- 1 root root  24M Feb 28  2003 ratings.dat
-rw-r----- 1 root root 5.5K Jan 29  2016 README
-rw-r----- 1 root root 132K Feb 28  2003 users.dat


In [4]:
# Step 3b: Load Data into Spark DataFrames
# ============================================================
# Ratings file:  UserID::MovieID::Rating::Timestamp
ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", FloatType(), True),
    StructField("timestamp", IntegerType(), True)
])

ratings_df = spark.read.csv(
    "ml-1m/ratings.dat",
    header=False,
    schema=ratings_schema,
    sep="::"
)

# Movies file:  MovieID::Title::Genres
movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

movies_df = spark.read.csv(
    "ml-1m/movies.dat",
    header=False,
    schema=movies_schema,
    sep="::"
)

# Cache for repeated use
ratings_df.cache()
movies_df.cache()

total_ratings = ratings_df.count()
total_movies = movies_df.count()
unique_users = ratings_df.select("userId").distinct().count()

print("✅ Data loaded successfully!")
print(f"   Total Ratings : {total_ratings:,}")
print(f"   Unique Users  : {unique_users:,}")
print(f"   Unique Movies : {total_movies:,}")


✅ Data loaded successfully!
   Total Ratings : 1,000,209
   Unique Users  : 6,040
   Unique Movies : 3,883


In [5]:
# Step 3c: Preview the Data
print("📊 Sample Ratings:")
ratings_df.show(5, truncate=False)

print("\n📽️ Sample Movies:")
movies_df.show(5, truncate=False)

print("\n📋 Schema:")
print("Ratings:", ratings_df.schema)
print("Movies :", movies_df.schema)


📊 Sample Ratings:
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1193   |5.0   |978300760|
|1     |661    |3.0   |978302109|
|1     |914    |3.0   |978301968|
|1     |3408   |4.0   |978300275|
|1     |2355   |5.0   |978824291|
+------+-------+------+---------+
only showing top 5 rows


📽️ Sample Movies:
+-------+----------------------------------+----------------------------+
|movieId|title                             |genres                      |
+-------+----------------------------------+----------------------------+
|1      |Toy Story (1995)                  |Animation|Children's|Comedy |
|2      |Jumanji (1995)                    |Adventure|Children's|Fantasy|
|3      |Grumpier Old Men (1995)           |Comedy|Romance              |
|4      |Waiting to Exhale (1995)          |Comedy|Drama                |
|5      |Father of the Bride Part II (1995)|Comedy                      |
+-------+-------------------------------

---

## Step 4: Exploratory Data Analysis (EDA)

Understanding the data distribution, user behavior, and movie characteristics before building the recommender model.


In [6]:
# Step 4a: Dataset Statistics
stats = ratings_df.agg(
    count("*").alias("total_ratings"),
    count("userId").alias("non_null_ratings"),
    mean("rating").alias("mean_rating"),
    min("rating").alias("min_rating"),
    max("rating").alias("max_rating")
).collect()[0]

unique_users = ratings_df.select("userId").distinct().count()
unique_movies = ratings_df.select("movieId").distinct().count()
sparsity = 1 - (stats["total_ratings"] / (unique_users * unique_movies))

print("📊 Dataset Statistics")
print("=" * 55)
print(f"  Total Ratings   : {stats['total_ratings']:,}")
print(f"  Unique Users    : {unique_users:,}")
print(f"  Unique Movies   : {unique_movies:,}")
print(f"  Rating Range    : {stats['min_rating']:.1f} – {stats['max_rating']:.1f}")
print(f"  Mean Rating     : {stats['mean_rating']:.3f}")
print(f"  Sparsity        : {sparsity*100:.2f}%")
print(f"  Density         : {(1-sparsity)*100:.4f}%")
print("=" * 55)
print("  Note: High sparsity (>99%) is typical for recommender systems.")


📊 Dataset Statistics
  Total Ratings   : 1,000,209
  Unique Users    : 6,040
  Unique Movies   : 3,706
  Rating Range    : 1.0 – 5.0
  Mean Rating     : 3.582
  Sparsity        : 95.53%
  Density         : 4.4684%
  Note: High sparsity (>99%) is typical for recommender systems.


In [7]:
# Step 4b: Rating Distribution (Interactive Plotly)
ratings_pd = ratings_df.select("rating").toPandas()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Rating Distribution", "Rating Value Counts"),
    specs=[[{"type": "histogram"}, {"type": "bar"}]]
)

# Histogram
fig.add_trace(
    go.Histogram(
        x=ratings_pd["rating"],
        nbinsx=10,
        marker_color="#636EFA",
        opacity=0.8,
        name="Count"
    ),
    row=1, col=1
)
fig.update_xaxes(title_text="Rating", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)

# Bar chart of value counts
value_counts = ratings_pd["rating"].value_counts().sort_index()
fig.add_trace(
    go.Bar(
        x=value_counts.index,
        y=value_counts.values,
        marker_color="#EF553B",
        opacity=0.8,
        text=[f"{v:,}" for v in value_counts.values],
        textposition="outside",
        name="Count"
    ),
    row=1, col=2
)
fig.update_xaxes(title_text="Rating", row=1, col=2, dtick=0.5)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.update_layout(
    title_text="<b>Rating Distribution Analysis</b>",
    title_font_size=16,
    showlegend=False,
    height=450,
    template="plotly_white"
)
fig.show()


Output hidden; open in https://colab.research.google.com to view.

In [8]:
# Step 4c: Genre Analysis
from pyspark.sql.functions import split, explode

# Explode genres (pipe-separated)
genre_counts = (
    movies_df
    .withColumn("genre", explode(split("genres", r"\|")))
    .groupBy("genre")
    .agg(count("*").alias("count"))
    .orderBy(desc("count"))
).toPandas()

fig = px.bar(
    genre_counts,
    x="count",
    y="genre",
    orientation="h",
    color="count",
    color_continuous_scale="Viridis",
    text="count"
)
fig.update_layout(
    title="<b>Movie Count by Genre</b>",
    title_font_size=16,
    xaxis_title="Number of Movies",
    yaxis_title="Genre",
    height=550,
    template="plotly_white",
    showlegend=False
)
fig.update_traces(textposition="outside")
fig.show()


In [9]:
# Step 4d: Top Rated Movies (min 100 ratings)
movie_stats = (
    ratings_df
    .groupBy("movieId")
    .agg(
        count("*").alias("num_ratings"),
        avg("rating").alias("avg_rating")
    )
    .filter(col("num_ratings") >= 100)
    .orderBy(desc("avg_rating"))
    .limit(15)
    .join(movies_df, "movieId", "inner")
    .select("title", "avg_rating", "num_ratings", "genres")
).toPandas()

fig = px.bar(
    movie_stats,
    x="avg_rating",
    y="title",
    orientation="h",
    color="avg_rating",
    color_continuous_scale="RdYlGn",
    text=[f"{r:.2f} ({c:,})" for r, c in zip(movie_stats["avg_rating"], movie_stats["num_ratings"])]
)
fig.update_layout(
    title="<b>Top 15 Highest Rated Movies (min 100 ratings)</b>",
    title_font_size=14,
    xaxis_title="Average Rating",
    yaxis_title="",
    height=550,
    template="plotly_white",
    showlegend=False
)
fig.update_traces(textposition="outside", textfont_size=10)
fig.show()


In [10]:
# Step 4e: User Activity Distribution
user_stats = (
    ratings_df
    .groupBy("userId")
    .agg(
        count("*").alias("num_ratings"),
        avg("rating").alias("avg_rating")
    )
).toPandas()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Ratings per User", "Average Rating per User"),
    specs=[[{"type": "histogram"}, {"type": "histogram"}]]
)

fig.add_trace(
    go.Histogram(
        x=user_stats["num_ratings"],
        nbinsx=60,
        marker_color="#636EFA",
        opacity=0.75,
        name="Users"
    ),
    row=1, col=1
)
fig.update_xaxes(title_text="Number of Ratings", row=1, col=1)
fig.update_yaxes(title_text="Number of Users", row=1, col=1)

fig.add_trace(
    go.Histogram(
        x=user_stats["avg_rating"],
        nbinsx=40,
        marker_color="#FECB52",
        opacity=0.75,
        name="Users"
    ),
    row=1, col=2
)
fig.update_xaxes(title_text="Average Rating Given", row=1, col=2)
fig.update_yaxes(title_text="Number of Users", row=1, col=2)

fig.update_layout(
    title_text="<b>User Activity Analysis</b>",
    title_font_size=16,
    showlegend=False,
    height=420,
    template="plotly_white"
)
fig.show()

print(f"\n📊 User Statistics:")
print(f"   Mean ratings per user : {user_stats['num_ratings'].mean():.1f}")
print(f"   Median ratings/user  : {user_stats['num_ratings'].median():.1f}")
print(f"   Max ratings by a user: {user_stats['num_ratings'].max():,}")
print(f"   Min ratings by a user: {user_stats['num_ratings'].min():,}")



📊 User Statistics:
   Mean ratings per user : 165.6
   Median ratings/user  : 96.0
   Max ratings by a user: 2,314
   Min ratings by a user: 20


---

## Step 5: Data Preprocessing

Preparing data for ALS matrix factorization. ALS requires integer user and item indices, so we use `StringIndexer` to create contiguous integer mappings.


In [11]:
# Step 5: Index userId and movieId for ALS
user_indexer = StringIndexer(
    inputCol="userId", outputCol="userIdx", handleInvalid="keep"
)
movie_indexer = StringIndexer(
    inputCol="movieId", outputCol="movieIdx", handleInvalid="keep"
)

# Fit on FULL data (ensures all IDs are mapped)
user_indexer_model = user_indexer.fit(ratings_df)
movie_indexer_model = movie_indexer.fit(ratings_df)

indexed_df = user_indexer_model.transform(ratings_df)
indexed_df = movie_indexer_model.transform(indexed_df)

# Cast to integer
indexed_df = indexed_df.withColumn("userIdx", col("userIdx").cast("int"))
indexed_df = indexed_df.withColumn("movieIdx", col("movieIdx").cast("int"))

indexed_df.cache()

# Verify indexing
num_idx_users = indexed_df.select("userIdx").distinct().count()
num_idx_movies = indexed_df.select("movieIdx").distinct().count()

print("✅ StringIndexer applied successfully!")
print(f"   Indexed Users  : {num_idx_users:,}")
print(f"   Indexed Movies : {num_idx_movies:,}")
indexed_df.show(5, truncate=False)


✅ StringIndexer applied successfully!
   Indexed Users  : 6,040
   Indexed Movies : 3,706
+------+-------+------+---------+-------+--------+
|userId|movieId|rating|timestamp|userIdx|movieIdx|
+------+-------+------+---------+-------+--------+
|1     |1193   |5.0   |978300760|4131   |43      |
|1     |661    |3.0   |978302109|4131   |585     |
|1     |914    |3.0   |978301968|4131   |461     |
|1     |3408   |4.0   |978300275|4131   |105     |
|1     |2355   |5.0   |978824291|4131   |47      |
+------+-------+------+---------+-------+--------+
only showing top 5 rows



In [12]:
# Step 5b: Train-Test Split (80/20)
# ============================================================
# No need to sample – 1M ratings fit comfortably in Colab memory.
# For the 20M dataset, add .sample(False, 0.1, seed=42) before split.

train_df, test_df = indexed_df.randomSplit([0.8, 0.2], seed=42)

# Cache for faster iteration
train_df.cache()
test_df.cache()

print("📊 Data Split:")
print("-" * 45)
print(f"   Training Set  : {train_df.count():>12,} ratings  (80%)")
print(f"   Test Set      : {test_df.count():>12,} ratings  (20%)")
print(f"   Total         : {train_df.count() + test_df.count():>12,} ratings")


📊 Data Split:
---------------------------------------------
   Training Set  :      800,177 ratings  (80%)
   Test Set      :      200,032 ratings  (20%)
   Total         :    1,000,209 ratings


---

## Step 6: Build ALS Recommender Model

Training the **Alternating Least Squares (ALS)** collaborative filtering model. ALS works by factorizing the user-item rating matrix into two lower-rank matrices (user factors and item factors), capturing latent features that explain observed ratings.

### Key ALS Parameters:
- **Rank**: Number of latent factors (higher = more expressive, but slower)
- **RegParam**: Regularization strength (prevents overfitting)
- **MaxIter**: Number of ALS iterations


In [13]:
# Step 6a: Configure ALS Model
als = ALS(
    maxIter=15,               # Number of ALS iterations
    regParam=0.1,             # L2 regularization
    rank=20,                  # Number of latent factors
    userCol="userIdx",        # Indexed user column
    itemCol="movieIdx",       # Indexed movie column
    ratingCol="rating",       # Rating column
    coldStartStrategy="drop", # Drop NaN predictions for unknown users/items
    nonnegative=True,         # Constrain factors to be non-negative
    implicitPrefs=False       # Explicit feedback (actual ratings)
)

print("🤖 ALS Model Configuration:")
print("-" * 45)
print(f"   Max Iterations  : {als.getMaxIter()}")
print(f"   Rank (Factors)  : {als.getRank()}")
print(f"   Regularization  : {als.getRegParam()}")
print(f"   Cold Start Strat: {als.getColdStartStrategy()}")
print(f"   Non-negative    : {als.getNonnegative()}")


🤖 ALS Model Configuration:
---------------------------------------------
   Max Iterations  : 15
   Rank (Factors)  : 20
   Regularization  : 0.1
   Cold Start Strat: drop
   Non-negative    : True


In [14]:
# Step 6b: Train the ALS Model
print("⏳ Training ALS Model...")
print("=" * 45)

start_time = time.time()
als_model = als.fit(train_df)
training_time = time.time() - start_time

print(f"✅ Training completed in {training_time:.2f} seconds!")
print(f"   Model Rank     : {als_model.rank}")
print(f"   User Factors   : {als_model.userFactors.count():,} users")
print(f"   Item Factors   : {als_model.itemFactors.count():,} movies")


⏳ Training ALS Model...
✅ Training completed in 57.46 seconds!
   Model Rank     : 20
   User Factors   : 6,040 users
   Item Factors   : 3,682 movies


---

## Step 7: Model Evaluation

Evaluating the model using multiple metrics:
- **RMSE** (Root Mean Squared Error) — penalizes large errors
- **MAE** (Mean Absolute Error) — average absolute deviation
- **Precision@K** — fraction of recommended items that are relevant
- **Recall@K** — fraction of relevant items that are recommended


In [15]:
# Step 7a: Generate Predictions & Compute RMSE / MAE
predictions = als_model.transform(test_df)

# Drop rows with NaN predictions (cold start)
valid_preds = predictions.filter(col("prediction").isNotNull())

evaluator_rmse = RegressionEvaluator(
    metricName="rmse", labelCol="rating", predictionCol="prediction"
)
evaluator_mae = RegressionEvaluator(
    metricName="mae", labelCol="rating", predictionCol="prediction"
)

rmse = evaluator_rmse.evaluate(valid_preds)
mae = evaluator_mae.evaluate(valid_preds)

print("📊 Model Performance Metrics")
print("=" * 50)
print(f"   RMSE : {rmse:.4f}")
print(f"   MAE  : {mae:.4f}")
print("-" * 50)
print(f"   💡 On average, predictions deviate by ±{mae:.2f} stars")
print(f"   💡 RMSE error is ~{rmse / 5 * 100:.1f}% of the 5-star scale")
print(f"   Valid predictions: {valid_preds.count():,} / {predictions.count():,}")


📊 Model Performance Metrics
   RMSE : 0.8627
   MAE  : 0.6894
--------------------------------------------------
   💡 On average, predictions deviate by ±0.69 stars
   💡 RMSE error is ~17.3% of the 5-star scale
   Valid predictions: 200,004 / 200,004


In [16]:
# Step 7b: Precision@K and Recall@K
# ============================================================
# Relevant = rating >= 3.5 (user liked the movie)
# K = 10 (top-10 recommendations per user)

K = 10
RELEVANCE_THRESHOLD = 3.5

def precision_recall_at_k(pred_df, k=10, threshold=3.5):
    """
    Computes Precision@K and Recall@K for ALS predictions.
    Relevant items are those with actual rating >= threshold.
    """
    # Filter to relevant items
    relevant = pred_df.filter(col("rating") >= threshold)

    # Per-user ground truth
    user_relevant = relevant.groupBy("userIdx").agg(
        collect_list("movieIdx").alias("relevant_items")
    )

    # Top-K predictions per user
    from pyspark.sql.window import Window
    from pyspark.sql.functions import row_number

    window = Window.partitionBy("userIdx").orderBy(desc("prediction"))
    top_k = (
        pred_df
        .withColumn("rank", row_number().over(window))
        .filter(col("rank") <= k)
        .groupBy("userIdx")
        .agg(collect_list("movieIdx").alias("recommended_items"))
    )

    # Join
    joined = user_relevant.join(top_k, "userIdx", "inner")

    # Compute precision & recall per user
    def count_overlap(rec_list, rel_list):
        if rec_list is None or rel_list is None:
            return 0
        return len(set(rec_list) & set(rel_list))

    results = joined.toPandas()
    results["hits"] = results.apply(
        lambda r: count_overlap(r["recommended_items"], r["relevant_items"]), axis=1
    )

    precision = results["hits"].sum() / (len(results) * k)
    recall = results["hits"].sum() / results["relevant_items"].apply(len).sum()

    return precision, recall, len(results)

precision_k, recall_k, n_users = precision_recall_at_k(valid_preds, k=K, threshold=RELEVANCE_THRESHOLD)

print(f"📊 Ranking Metrics (K={K}, relevance >= {RELEVANCE_THRESHOLD})")
print("=" * 55)
print(f"   Precision@{K} : {precision_k:.4f}  ({precision_k*100:.1f}%)")
print(f"   Recall@{K}    : {recall_k:.4f}  ({recall_k*100:.1f}%)")
print(f"   Evaluated Users: {n_users:,}")


📊 Ranking Metrics (K=10, relevance >= 3.5)
   Precision@10 : 0.6980  (69.8%)
   Recall@10    : 0.3632  (36.3%)
   Evaluated Users: 5,976


In [17]:
# Step 7c: Visualize Prediction Quality
preds_pd = valid_preds.select("rating", "prediction").toPandas()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Actual vs Predicted", "Residual Distribution"),
    specs=[[{"type": "scatter"}, {"type": "histogram"}]]
)

# Scatter: Actual vs Predicted (sample for performance)
sample = preds_pd.sample(__builtins__.min(10000, len(preds_pd)), random_state=42)
fig.add_trace(
    go.Scatter(
        x=sample["rating"],
        y=sample["prediction"],
        mode="markers",
        marker=dict(size=4, opacity=0.3, color="#636EFA"),
        name="Predictions"
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=[0.5, 5], y=[0.5, 5],
        mode="lines",
        line=dict(color="red", dash="dash", width=2),
        name="Perfect"
    ),
    row=1, col=1
)
fig.update_xaxes(title_text="Actual Rating", range=[0.5, 5.0], row=1, col=1)
fig.update_yaxes(title_text="Predicted Rating", range=[0.5, 5.0], row=1, col=1)

# Residual histogram
residuals = preds_pd["prediction"] - preds_pd["rating"]
fig.add_trace(
    go.Histogram(
        x=residuals,
        nbinsx=60,
        marker_color="#AB63FA",
        opacity=0.8,
        name="Residuals"
    ),
    row=1, col=2
)
fig.add_vline(
    x=0, line_dash="dash", line_color="red", line_width=2, row=1, col=2
)
fig.update_xaxes(title_text="Residual (Predicted − Actual)", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.update_layout(
    title_text="<b>Prediction Quality Analysis</b>",
    title_font_size=16,
    showlegend=True,
    height=420,
    template="plotly_white"
)
fig.show()


---

## Step 8: Generate Recommendations

Creating personalized movie recommendations using the trained ALS model. The model predicts ratings for unseen user-movie pairs and returns the top-N highest scored movies for each user.


In [18]:
# Step 8a: Build Recommendation Function
user_idx_map = (
    indexed_df.select("userId", "userIdx").distinct()
)
movie_idx_map = (
    indexed_df.select("movieId", "movieIdx").distinct()
)

def get_recommendations(user_id, num_recs=10):
    """
    Get top-N movie recommendations for a given user.

    Args:
        user_id      : Raw userId from the dataset
        num_recs     : Number of recommendations to return

    Returns:
        Spark DataFrame with recommended movies (title, genres, predicted rating)
    """
    # Lookup user index
    user_row = user_idx_map.filter(col("userId") == user_id).first()
    if user_row is None:
        print(f"❌ User {user_id} not found in the index.")
        return None

    user_index = user_row["userIdx"]

    # Movies already rated by this user
    rated_movie_ids = (
        indexed_df
        .filter(col("userIdx") == user_index)
        .select("movieIdx")
        .distinct()
    )

    # All candidate movies minus already-rated
    candidates = movie_idx_map.select("movieIdx").distinct() \
        .subtract(rated_movie_ids) \
        .withColumn("userIdx", lit(user_index))

    # Predict
    scored = als_model.transform(candidates) \
        .filter(col("prediction").isNotNull()) \
        .orderBy(desc("prediction")) \
        .limit(num_recs)

    # Join back to movie metadata
    recs = (
        scored
        .join(movie_idx_map, "movieIdx")
        .join(movies_df, "movieId")
        .select("title", "genres", "prediction")
    )
    return recs

def show_user_ratings(user_id, n=10):
    """Show top-N ratings from a user."""
    return (
        indexed_df
        .filter(col("userId") == user_id)
        .join(movies_df, "movieId")
        .select("title", "genres", "rating")
        .orderBy(desc("rating"))
        .limit(n)
    )

print("✅ Recommendation engine ready!")


✅ Recommendation engine ready!


In [19]:
# Step 8b: Recommendations for User 1
user_id = 1

print(f"🎬 Movies rated by User {user_id} (highest first):")
print("-" * 70)
show_user_ratings(user_id, 10).show(truncate=False)

print(f"\n💡 Top 10 Recommendations for User {user_id}:")
print("-" * 70)
recs = get_recommendations(user_id, 10)
if recs:
    recs.show(truncate=False)


🎬 Movies rated by User 1 (highest first):
----------------------------------------------------------------------
+--------------------------------------+------------------------------------+------+
|title                                 |genres                              |rating|
+--------------------------------------+------------------------------------+------+
|Schindler's List (1993)               |Drama|War                           |5.0   |
|Saving Private Ryan (1998)            |Action|Drama|War                    |5.0   |
|Pocahontas (1995)                     |Animation|Children's|Musical|Romance|5.0   |
|Back to the Future (1985)             |Comedy|Sci-Fi                       |5.0   |
|Last Days of Disco, The (1998)        |Drama                               |5.0   |
|Ben-Hur (1959)                        |Action|Adventure|Drama              |5.0   |
|Cinderella (1950)                     |Animation|Children's|Musical        |5.0   |
|One Flew Over the Cuckoo's Nest (197

In [20]:
# Step 8c: Try Different Users
# ============================================================
# Change user_id to explore different recommendation profiles
# Try: 1, 42, 100, 500, 2000, 3000

user_id = 42

print(f"🎬 Top 5 Movies Rated by User {user_id}:")
print("-" * 70)
show_user_ratings(user_id, 5).show(truncate=False)

print(f"\n💡 Top 5 Recommendations for User {user_id}:")
print("-" * 70)
recs = get_recommendations(user_id, 5)
if recs:
    recs_pd = recs.toPandas()
    for _, row in recs_pd.iterrows():
        print(f"  📽️  {row['title']}")
        print(f"      Genres: {row['genres']}")
        print(f"      Predicted Rating: ⭐ {row['prediction']:.2f}\n")


🎬 Top 5 Movies Rated by User 42:
----------------------------------------------------------------------
+-----------------------------------------------------+---------------------------------+------+
|title                                                |genres                           |rating|
+-----------------------------------------------------+---------------------------------+------+
|Groundhog Day (1993)                                 |Comedy|Romance                   |5.0   |
|Back to the Future (1985)                            |Comedy|Sci-Fi                    |5.0   |
|Star Wars: Episode V - The Empire Strikes Back (1980)|Action|Adventure|Drama|Sci-Fi|War|5.0   |
|Better Off Dead... (1985)                            |Comedy                           |5.0   |
|Raiders of the Lost Ark (1981)                       |Action|Adventure                 |5.0   |
+-----------------------------------------------------+---------------------------------+------+


💡 Top 5 Recommendatio

In [21]:
# Step 8d: Batch Recommendations for Multiple Users
print("📊 Generating batch recommendations for 10 random users...")

sample_users = user_idx_map.sample(False, 0.005, seed=42).limit(10)
batch_recs = als_model.recommendForUserSubset(sample_users, 5)

# Flatten the array of recommendations
batch_flat = (
    batch_recs
    .withColumn("rec", explode("recommendations"))
    .select(
        col("userIdx"),
        col("rec.movieIdx").alias("movieIdx"),
        col("rec.rating").alias("prediction")
    )
    .join(user_idx_map, "userIdx")
    .join(movie_idx_map, "movieIdx")
    .join(movies_df, "movieId")
    .select("userId", "title", "prediction")
    .orderBy("userId", desc("prediction"))
)

print(f"\n✅ Generated {batch_flat.count()} recommendations for {sample_users.count()} users")
batch_flat.show(20, truncate=False)


📊 Generating batch recommendations for 10 random users...

✅ Generated 50 recommendations for 10 users
+------+--------------------------------------------------+----------+
|userId|title                                             |prediction|
+------+--------------------------------------------------+----------+
|415   |Foreign Student (1994)                            |4.6849713 |
|415   |Hearts and Minds (1996)                           |4.599165  |
|415   |For All Mankind (1989)                            |4.588475  |
|415   |Inheritors, The (Die Siebtelbauern) (1998)        |4.5748534 |
|415   |Apple, The (Sib) (1998)                           |4.5585356 |
|633   |Inheritors, The (Die Siebtelbauern) (1998)        |4.865328  |
|633   |Return with Honor (1998)                          |4.8214374 |
|633   |Hearts and Minds (1996)                           |4.6622005 |
|633   |American Beauty (1999)                            |4.6573315 |
|633   |Fargo (1996)                         

---

## Step 9: Model Comparison & Hyperparameter Tuning

Testing different ALS configurations to find the optimal combination of rank and regularization parameter. We evaluate each configuration using RMSE and MAE on the held-out test set.


In [22]:
# Step 9a: Grid Search over Hyperparameters
configs = [
    {"rank": 5,  "regParam": 0.1},
    {"rank": 10, "regParam": 0.1},
    {"rank": 20, "regParam": 0.1},
    {"rank": 50, "regParam": 0.1},
    {"rank": 10, "regParam": 0.05},
    {"rank": 10, "regParam": 0.2},
    {"rank": 20, "regParam": 0.05},
    {"rank": 20, "regParam": 0.2},
]

results = []

print("🔬 Hyperparameter Tuning")
print("=" * 65)

for i, cfg in enumerate(configs):
    print(f"\n  [{i+1}/{len(configs)}] Rank={cfg['rank']:>2}, RegParam={cfg['regParam']}")

    als_tune = ALS(
        maxIter=10,
        regParam=cfg["regParam"],
        rank=cfg["rank"],
        userCol="userIdx",
        itemCol="movieIdx",
        ratingCol="rating",
        coldStartStrategy="drop",
        nonnegative=True
    )

    t0 = time.time()
    model = als_tune.fit(train_df)
    preds = model.transform(test_df).filter(col("prediction").isNotNull())

    rmse_val = evaluator_rmse.evaluate(preds)
    mae_val = evaluator_mae.evaluate(preds)
    elapsed = time.time() - t0

    results.append({
        "rank": cfg["rank"],
        "regParam": cfg["regParam"],
        "RMSE": rmse_val,
        "MAE": mae_val,
        "time": elapsed
    })
    print(f"      RMSE={rmse_val:.4f}, MAE={mae_val:.4f}, Time={elapsed:.1f}s")

print("\n✅ Tuning complete!")


🔬 Hyperparameter Tuning

  [1/8] Rank= 5, RegParam=0.1
      RMSE=0.8816, MAE=0.7074, Time=28.2s

  [2/8] Rank=10, RegParam=0.1
      RMSE=0.8720, MAE=0.6991, Time=29.1s

  [3/8] Rank=20, RegParam=0.1
      RMSE=0.8683, MAE=0.6961, Time=30.7s

  [4/8] Rank=50, RegParam=0.1
      RMSE=0.8660, MAE=0.6937, Time=47.7s

  [5/8] Rank=10, RegParam=0.05
      RMSE=0.8599, MAE=0.6816, Time=24.4s

  [6/8] Rank=10, RegParam=0.2
      RMSE=0.9206, MAE=0.7457, Time=22.2s

  [7/8] Rank=20, RegParam=0.05
      RMSE=0.8581, MAE=0.6786, Time=31.6s

  [8/8] Rank=20, RegParam=0.2
      RMSE=0.9201, MAE=0.7453, Time=28.8s

✅ Tuning complete!


In [23]:
# Step 9b: Visualize Tuning Results
results_df = pd.DataFrame(results)
results_df["config"] = (
    "R" + results_df["rank"].astype(str) + ",reg" + results_df["regParam"].astype(str)
)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("RMSE by Configuration", "MAE by Configuration"),
    specs=[[{"type": "bar"}, {"type": "bar"}]]
)

colors = px.colors.qualitative.Pastel * 2

fig.add_trace(
    go.Bar(
        x=results_df["config"],
        y=results_df["RMSE"],
        marker_color=colors[:len(results_df)],
        text=[f"{v:.4f}" for v in results_df["RMSE"]],
        textposition="outside",
        textfont_size=9,
        name="RMSE"
    ),
    row=1, col=1
)
fig.update_yaxes(title_text="RMSE", range=[0.8, 1.0], row=1, col=1)

fig.add_trace(
    go.Bar(
        x=results_df["config"],
        y=results_df["MAE"],
        marker_color=colors[:len(results_df)],
        text=[f"{v:.4f}" for v in results_df["MAE"]],
        textposition="outside",
        textfont_size=9,
        name="MAE"
    ),
    row=1, col=2
)
fig.update_yaxes(title_text="MAE", range=[0.6, 0.85], row=1, col=2)

fig.update_layout(
    title_text="<b>Hyperparameter Tuning Results</b>",
    title_font_size=16,
    xaxis_tickangle=45,
    height=420,
    template="plotly_white",
    showlegend=False
)
fig.show()

# Best configuration
best_idx = results_df["RMSE"].idxmin()
best = results_df.loc[best_idx]
print(f"\n🏆 Best Configuration:")
print(f"   Rank={int(best['rank'])}, RegParam={best['regParam']}")
print(f"   RMSE={best['RMSE']:.4f}, MAE={best['MAE']:.4f}")



🏆 Best Configuration:
   Rank=20, RegParam=0.05
   RMSE=0.8581, MAE=0.6786


---

## Step 10: Cold Start Problem

A common challenge in recommender systems: how to recommend movies to **new users** (with no rating history) or recommend **new movies** (with no ratings)?


In [24]:
# Step 10: Cold Start Strategy
print("❄️  Cold Start Problem & Solutions")
print("=" * 55)

# Strategy 1: Recommend globally popular movies
popular = (
    ratings_df
    .groupBy("movieId")
    .agg(
        count("*").alias("num_ratings"),
        avg("rating").alias("avg_rating")
    )
    .filter(col("num_ratings") >= 100)
    .join(movies_df, "movieId")
    .select("title", "genres", "num_ratings", "avg_rating")
    .orderBy(desc("avg_rating"))
    .limit(10)
)

print("\n🌟 Strategy 1 – Top 10 Popular Movies (for new users):")
print("-" * 55)
popular.show(truncate=False)

print("\n💡 Cold Start Strategies:")
print("   1. New Users  → Recommend popular / highly-rated movies")
print("   2. New Movies → Use content-based features (genre, director, etc.)")
print("   3. Hybrid     → Combine collaborative + content-based filtering")
print("   4. Exploration → Use multi-armed bandits to explore user preferences")


❄️  Cold Start Problem & Solutions

🌟 Strategy 1 – Top 10 Popular Movies (for new users):
-------------------------------------------------------
+-------------------------------------------------------------------+-------------------------+-----------+-----------------+
|title                                                              |genres                   |num_ratings|avg_rating       |
+-------------------------------------------------------------------+-------------------------+-----------+-----------------+
|Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)|Action|Drama             |628        |4.560509554140127|
|Shawshank Redemption, The (1994)                                   |Drama                    |2227       |4.554557700942973|
|Godfather, The (1972)                                              |Action|Crime|Drama       |2223       |4.524966261808367|
|Close Shave, A (1995)                                              |Animation|Comedy|Thriller|657

---

## Step 11: Save & Load Model

Persisting the trained ALS model and index mappings for future use, enabling deployment without retraining.


In [25]:
# Step 11a: Save Model & Mappings
model_path = "models/als_movie_recommender"
os.makedirs("models", exist_ok=True)

# Use overwrite() to prevent 'Path already exists' error
als_model.write().overwrite().save(model_path)

# Save index mappings
user_idx_map.toPandas().to_csv("models/user_mapping.csv", index=False)
movie_idx_map.toPandas().to_csv("models/movie_mapping.csv", index=False)

print("✅ Model and mappings saved successfully!")
print("\n📁 Saved artifacts:")
for root, dirs, files in os.walk("models"):
    level = root.replace("models", "").count(os.sep)
    indent = "   " * (level + 1)
    for f in files:
        size = os.path.getsize(os.path.join(root, f))
        print(f"{indent}📁 {f}  ({size/1024:.1f} KB)")

✅ Model and mappings saved successfully!

📁 Saved artifacts:
   📁 user_mapping.csv  (56.8 KB)
   📁 movie_mapping.csv  (34.1 KB)
         📁 part-00000-ce3891c9-8451-40b7-bb85-6f3be1865a1a-c000.snappy.parquet  (29.8 KB)
         📁 part-00004-ce3891c9-8451-40b7-bb85-6f3be1865a1a-c000.snappy.parquet  (29.8 KB)
         📁 part-00006-ce3891c9-8451-40b7-bb85-6f3be1865a1a-c000.snappy.parquet  (29.9 KB)
         📁 ._SUCCESS.crc  (0.0 KB)
         📁 _SUCCESS  (0.0 KB)
         📁 .part-00001-ce3891c9-8451-40b7-bb85-6f3be1865a1a-c000.snappy.parquet.crc  (0.2 KB)
         📁 .part-00004-ce3891c9-8451-40b7-bb85-6f3be1865a1a-c000.snappy.parquet.crc  (0.2 KB)
         📁 .part-00007-ce3891c9-8451-40b7-bb85-6f3be1865a1a-c000.snappy.parquet.crc  (0.2 KB)
         📁 .part-00003-ce3891c9-8451-40b7-bb85-6f3be1865a1a-c000.snappy.parquet.crc  (0.2 KB)
         📁 .part-00006-ce3891c9-8451-40b7-bb85-6f3be1865a1a-c000.snappy.parquet.crc  (0.2 KB)
         📁 part-00009-ce3891c9-8451-40b7-bb85-6f3be1865a1a-c000.sna

In [26]:
# Step 11b: Load & Test Saved Model
print("🔄 Loading saved model...")

loaded_model = ALSModel.load(model_path)

# Quick test
test_user_row = user_idx_map.first()
test_user_idx = test_user_row["userIdx"]
test_user_raw = test_user_row["userId"]

test_recs = loaded_model.recommendForUserSubset(
    spark.createDataFrame([(test_user_idx,)], ["userIdx"]),
    5
)

print(f"✅ Model loaded successfully!")
print(f"\n💡 Sample recommendations for User {test_user_raw}:")
test_recs.show(truncate=False)


🔄 Loading saved model...
✅ Model loaded successfully!

💡 Sample recommendations for User 1:
+-------+------------------------------------------------------------------------------------+
|userIdx|recommendations                                                                     |
+-------+------------------------------------------------------------------------------------+
|4131   |[{3570, 5.333554}, {23, 4.544304}, {17, 4.540805}, {239, 4.432534}, {44, 4.4084682}]|
+-------+------------------------------------------------------------------------------------+

